# Decompositions

## Why this matters

Decomposing a matrix means writing it as a product of simpler pieces. A raw matrix mixes rotation, stretching and reflection together and is hard to read. Think of splitting white light with a prism: the decomposition shows which directions a matrix cares about and by how much. Eigendecomposition and the singular value decomposition (SVD) pull those effects apart. The singular values are the stretch amounts, ranked largest to smallest; the big ones are where the data lives, the small ones are near noise. Read them to decide how many directions to keep.

Least squares is where these ideas fit a line to data. It has one dial worth naming early: forming the normal equation squares the condition number, so a matrix with condition number 1e6 solves at an effective 1e12 and loses most of its precision. Ridge regression is the fix. It adds a penalty on large weights, set by a strength lambda (a typical start is 0.1 to 1.0), which lifts the small singular values off zero and steadies the solution.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression

# One seeded generator for every random array below.
# Fixed seed keeps the figures and the quoted numbers (errors, variance shares) stable across runs.
rng = np.random.default_rng(42)
np.set_printoptions(precision=3, suppress=True)

## 1. Intuition

Push a vector through a matrix and it usually comes out pointing somewhere new. A few special vectors do not turn at all, they only get longer or shorter. Those are the eigenvectors, and the factor they scale by is the eigenvalue. They are the natural axes of the transformation.

SVD generalises this to any matrix, including rectangular ones with no eigenvectors. Every matrix is three clean steps: a rotation, a stretch along perpendicular axes, then another rotation. The stretch amounts are the singular values, ranked largest to smallest. The big ones are the directions the data mostly lives in, the small ones are close to noise. That ranking makes SVD the engine behind PCA, compression and the normal equation below.

## 2. Maths

**Eigen:** $Av = \lambda v$, so $A$ acting on $v$ is the same as scaling $v$ by $\lambda$. For a symmetric $A$, such as any covariance matrix, the eigenvectors are perpendicular and $A = Q \Lambda Q^T$ with $Q$ orthogonal. Most matrices that matter in ML are symmetric.

**SVD:** $X = U \Sigma V^T$. $U$ and $V$ have orthonormal columns (the two rotations), $\Sigma$ is diagonal with singular values $\sigma_1 \ge \sigma_2 \ge \dots \ge 0$ (the stretch). It exists for every matrix.

**Link:** the singular values of $X$ squared are the eigenvalues of $X^T X$. SVD and eigendecomposition are one idea seen from two sides.

**Low-rank approximation:** keep the top $k$ singular values and drop the rest. By Eckart-Young this is the best rank-$k$ approximation in the least-squares sense.

**Least squares:** minimise $\lVert Xw - y \rVert^2$. Setting the gradient to zero gives the normal equation $X^T X w = X^T y$, one linear system to solve.

## 3. Cost

A full SVD of an $n \times d$ matrix costs about $O(n d^2)$. The payoff is storage. Keeping $k$ components stores $k(n + d + 1)$ numbers instead of $nd$. For a 1000 by 1000 image at $k = 20$ that is 40,020 numbers instead of 1,000,000, a 25x saving, and the image still looks almost the same.

The normal equation needs $X^T X$ at $O(n d^2)$, then a solve at $O(d^3)$. Its real cost is not speed but numerical stability, which section 6 gets to.

## 4. From scratch

### Eigenvectors by picture

A symmetric matrix applied to the unit circle gives an ellipse. The eigenvectors are the ellipse axes.

In [ ]:
# Fixed symmetric 2x2 test matrix. Symmetric is chosen on purpose: eigh guarantees real
# eigenvalues and perpendicular eigenvectors, so the picture is a clean ellipse with axes
# along the eigenvectors. Covariance matrices in ML are symmetric too, so this is the useful case.
A = np.array([[3.0, 1.0], [1.0, 3.0]])
vals, vecs = np.linalg.eigh(A)
print("eigenvalues:", vals)
print("eigenvectors (columns):\n", vecs)

theta = np.linspace(0, 2 * np.pi, 200)
circle = np.vstack([np.cos(theta), np.sin(theta)])
fig, ax = plt.subplots(figsize=(4, 4))
ax.plot(*circle, alpha=0.4, label="unit circle")
ax.plot(*(A @ circle), label="A applied")
for lam, v in zip(vals, vecs.T):
    ax.annotate("", xy=lam * v, xytext=(0, 0), arrowprops={"arrowstyle": "->", "color": "C3"})
ax.set_aspect("equal")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

v = vecs[:, 1]
print("A v == lambda v:", np.allclose(A @ v, vals[1] * v))

The ellipse axes have lengths 4 and 2, exactly the two eigenvalues. The matrix stretches space by 4 along one eigenvector and by 2 along the other, so the circle becomes that ellipse. Reading an eigenvalue off a picture like this is the fastest way to build intuition.

### Power iteration

The dominant eigenvector falls out of repeated multiplication, no eigen-solver needed. Apply $A$, rescale to unit length, repeat. Each step amplifies the largest-eigenvalue direction more than the others, so after enough steps everything else has washed out. This is the seed of the algorithm Google's original PageRank ran on the web link matrix.

In [ ]:
def power_iteration(A, steps=50, seed=0):
    v = np.random.default_rng(seed).normal(size=A.shape[0])
    for _ in range(steps):
        v = A @ v
        v /= np.linalg.norm(v)
    return v @ A @ v, v


lam, v = power_iteration(A)
print("dominant eigenvalue:", round(lam, 4), "vector:", v)

### SVD and low-rank approximation

Build a smooth 60 by 60 'image' plus a little noise. Rank 3 captures nearly all of it.

In [ ]:
# Synthetic 60x60 "image", float64. Built from a few sine/cosine and polynomial terms so it
# is smooth and genuinely low rank: a product of 1D functions has rank 1, a sum of a few has
# low rank. That is why a handful of singular values capture it. A small 0.01 noise term adds
# the long tail of near-zero singular values. Seeded so the printed errors stay fixed.
x = np.linspace(0, 1, 60)
img = np.outer(np.sin(3 * x), np.cos(2 * x)) + 0.5 * np.outer(x, x**2)
img += 0.01 * rng.normal(size=img.shape)

U, s, Vt = np.linalg.svd(img, full_matrices=False)
print("top singular values:", s[:5])


def rank_k(U, s, Vt, k):
    return (U[:, :k] * s[:k]) @ Vt[:k, :]


for k in (1, 2, 3, 10):
    err = np.linalg.norm(img - rank_k(U, s, Vt, k)) / np.linalg.norm(img)
    print(f"rank {k:2d}: relative error {err:.4f}")

# check the link to eigenvalues of X^T X
eig = np.sort(np.linalg.eigvalsh(img.T @ img))[::-1]
print("sigma^2 == eig(X^T X):", np.allclose(s**2, eig))

Two components already give an error of about 2%, close to the noise floor we added. The rest of the singular values describe noise, not signal, so keeping them buys nothing. Real data behaves the same way almost every time: a handful of directions carry most of the structure and the long tail is noise. That is the fact that makes dimensionality reduction work.

### PCA pointer

PCA is the SVD of the centred data matrix. Subtract the mean from each column, take the SVD, and the right singular vectors are the principal components, the directions of greatest variance. Each explains variance $\sigma_i^2/(n-1)$. Section 04 builds this properly. Here is the one-line version:

In [ ]:
# Correlated 2D sample: 200 points, float64. Standard normals passed through a fixed mixing
# matrix so the cloud stretches more along one direction than the other, giving PCA something
# to find. Xc centres it (mean 0 per column), which PCA needs. Seeded so variance shares hold.
X = rng.normal(size=(200, 2)) @ np.array([[2.0, 0.0], [1.0, 0.5]])
Xc = X - X.mean(axis=0)
_, s_x, Vt_x = np.linalg.svd(Xc, full_matrices=False)
print("principal directions:\n", Vt_x)
print("variance explained:", s_x**2 / (s_x**2).sum())

### Least squares: the normal equation

Fit $y = 2 + 3x$ with noise. Add a column of ones so the intercept is just another weight.

In [ ]:
# Regression sample from a known line y = 2 + 3x plus noise, so we can check the recovered
# weights against (2, 3). n = 100 points. X is shape (100, 2): a column of ones for the
# intercept and x1 drawn uniform on [-1, 1]. Noise is Gaussian at scale 0.3. Seeded so the
# fitted weights and plot repeat.
n = 100
x1 = rng.uniform(-1, 1, size=n)
X = np.column_stack([np.ones(n), x1])
y = 2.0 + 3.0 * x1 + 0.3 * rng.normal(size=n)

w_normal = np.linalg.solve(X.T @ X, X.T @ y)
print("normal equation:", w_normal)

# via SVD: w = V diag(1/s) U^T y
U, s, Vt = np.linalg.svd(X, full_matrices=False)
w_svd = Vt.T @ ((U.T @ y) / s)
print("SVD solution:   ", w_svd)

fig, ax = plt.subplots(figsize=(4, 3))
ax.scatter(x1, y, s=8, alpha=0.5)
xs = np.array([-1, 1])
ax.plot(xs, w_normal[0] + w_normal[1] * xs, "C1")
plt.show()

## 5. Library version

In [ ]:
w_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)
print("np.linalg.lstsq:", w_lstsq)

lr = LinearRegression().fit(x1.reshape(-1, 1), y)
print("sklearn:", lr.intercept_, lr.coef_)
print(
    "all agree:",
    np.allclose(w_normal, w_lstsq) and np.allclose(w_lstsq, [lr.intercept_, *lr.coef_]),
)

## 6. What breaks

The normal equation has a hidden trap. Forming $X^T X$ squares the condition number: $\kappa(X^T X) = \kappa(X)^2$. A matrix that was mildly awkward to work with becomes badly awkward, because squaring a large number makes it much larger. When two features are nearly collinear, $\kappa(X)$ is already big and squaring it destroys most of your precision.

In [ ]:
# Near-collinear sample to show the normal equation's instability. n = 100 (from above).
# Two feature columns that are almost identical: the second is the first plus a tiny 1e-6
# Gaussian wobble. That 1e-6 is deliberate: large enough to keep the columns independent (a
# unique solution exists) but small enough that X^T X is badly conditioned. Seeded for repeatable weights.
x_a = rng.uniform(-1, 1, size=n)
X_bad = np.column_stack([x_a, x_a + 1e-6 * rng.normal(size=n)])
y_bad = x_a + 0.1 * rng.normal(size=n)

print("cond(X):    ", f"{np.linalg.cond(X_bad):.2e}")
print("cond(X^T X):", f"{np.linalg.cond(X_bad.T @ X_bad):.2e}")
print("normal eq weights:", np.linalg.solve(X_bad.T @ X_bad, X_bad.T @ y_bad))
print("lstsq weights:    ", np.linalg.lstsq(X_bad, y_bad, rcond=None)[0])

The two features are almost the same column, so near-opposite weights on them barely change the predictions. The tiny 1e-6 difference keeps the columns independent, so the least-squares solution is still unique, but it is extremely sensitive: a small change in the data swings the coefficients a long way. That is why the solver returns large weights of opposite sign that nearly cancel, a fit that is fragile and overfits. A ridge penalty stabilises those coefficients by preferring small weights, and section 03 derives it from a Gaussian prior. The practical rule stands alone: use the library solver, which works from the SVD and never squares the condition number, instead of inverting $X^T X$ yourself.